In [1]:
import chemprop as cp
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, GroupShuffleSplit

import deltaprop
from tuner import tune_model
from utils import (
    RANDOM_SEED,
    generate_features,
    get_scaffold,
    mol_to_inchi,
    standardize,
)

import wandb

wandb.login(key="cf344975eb80edf6f0d52af80528cc6094234caf")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/rahul/.netrc
wandb: Currently logged in as: rahul-e-dev to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
df = pd.read_excel("./evaluate_tba/GSK_TBA_AH_JSFedit070425.xlsx")
df["target"] = df['parent_remaining_24'].round(1) / 100 + df["metabolite_detected"]
df = df.loc[:, ["smiles", "parent_remaining_24", "metabolite_detected", "target"]]
df["mol"] = df["smiles"].map(standardize)
df = df.dropna(subset=["mol"])
df["inchi"] = df["mol"].map(mol_to_inchi)
df = df.groupby(["inchi"]).filter(lambda x: len(x) == 1).reset_index(drop=True)

df = generate_features(df)

clusters, _ = pd.factorize(df["mol"].map(get_scaffold))
df["cluster"] = pd.Series(clusters)

df = df.drop("inchi", axis=1)

In [ ]:
def get_molecule_datapoint(row):
    feat_entry_names = [f for f in row.index if f.startswith("feat")]
    feat_array = pd.to_numeric(row[feat_entry_names], errors="coerce")
    return cp.data.MoleculeDatapoint(
        mol=row["mol"], y=np.array([row["target"]]), x_d=feat_array.to_numpy()
    )


df["mol_dp"] = df.apply(get_molecule_datapoint, axis=1)

In [ ]:
def generate_repeated_5x5_splits(df):
    rng = np.random.RandomState(RANDOM_SEED)
    for outer_idx in range(5):
        randint = rng.randint(low=0, high=32767)
        cv = GroupKFold(n_splits=5, shuffle=True, random_state=randint)  # type: ignore
        for inner_idx, (train_val_idxs, test_idxs) in enumerate(
            cv.split(df, groups=df["cluster"])
        ):
            train_val_df = df.loc[train_val_idxs].reset_index(drop=True)
            test_df = df.loc[test_idxs].reset_index(drop=True)

            train_idx, val_idxs = next(
                GroupShuffleSplit(1, random_state=randint).split(
                    train_val_df, groups=train_val_df["cluster"]
                )
            )

            train_df = train_val_df.loc[train_idx].reset_index(drop=True)
            val_df = train_val_df.loc[val_idxs].reset_index(drop=True)

            yield (outer_idx, inner_idx), (train_df, val_df, test_df)


def generate_repeated_5x2_splits(df):
    rng = np.random.RandomState(RANDOM_SEED)
    for outer_idx in range(5):
        randint = rng.randint(low=0, high=32767)
        cv = GroupKFold(n_splits=5, shuffle=True, random_state=randint)  # type: ignore
        for inner_idx, (train_idxs, val_test_idxs) in enumerate(
            cv.split(df, groups=df["cluster"])
        ):
            train_df = df.loc[train_idxs].reset_index(drop=True)
            val_test_df = df.loc[val_test_idxs].reset_index(drop=True)

            val_idxs, test_idxs = next(
                GroupShuffleSplit(1, test_size=0.5, random_state=randint).split(
                    val_test_df, groups=val_test_df["cluster"]
                )
            )

            val_df = val_test_df.loc[val_idxs].reset_index(drop=True)
            test_df = val_test_df.loc[test_idxs].reset_index(drop=True)

            yield (outer_idx, inner_idx), (train_df, val_df, test_df)

In [ ]:
def prepare_mol_datasets(train_df, val_df, test_df):
    featurizer = cp.featurizers.SimpleMoleculeMolGraphFeaturizer()
    train_mol_dataset = cp.data.MoleculeDataset(
        train_df["mol_dp"], featurizer=featurizer
    )
    val_mol_dataset = cp.data.MoleculeDataset(val_df["mol_dp"], featurizer=featurizer)
    test_mol_dataset = cp.data.MoleculeDataset(test_df["mol_dp"], featurizer=featurizer)

    x_d_scaler = train_mol_dataset.normalize_inputs("X_d")
    val_mol_dataset.normalize_inputs("X_d", x_d_scaler)
    # test_mol_dataset.normalize_inputs("X_d", x_d_scaler)

    train_mol_dataset.cache = True
    val_mol_dataset.cache = True
    # test_mol_dataset.cache = True

    return train_mol_dataset, val_mol_dataset, test_mol_dataset, x_d_scaler

In [ ]:
run = wandb.init(project="evaluate_tba", tags=["deltaprop"])
run.mark_preempting()

for idx, (split_idxs, split) in enumerate(generate_repeated_5x2_splits(df)):
    outer_idx, inner_idx = split_idxs
    train_df, val_df, test_df = split

    train_mol_ds, val_mol_ds, test_mol_ds, X_d_scaler = prepare_mol_datasets(
        train_df, val_df, test_df
    )

    best_config = tune_model(
        tune_func=deltaprop.tune_func,
        search_space=deltaprop.search_space,
        train_mol_ds=train_mol_ds,
        val_mol_ds=val_mol_ds,
        X_d_scaler=X_d_scaler,
        batch_size=16,
        max_epochs=20,
        binary_threshold=1.5,
        max_concurrent=1,
        num_samples=4,
    )

    model = deltaprop.train_func(
        config=best_config,
        train_mol_ds=train_mol_ds,
        val_mol_ds=val_mol_ds,
        X_d_scaler=X_d_scaler,
        binary_threshold=1.5,
        batch_size=16,
    )

    preds, pred_probs, labels = deltaprop.predict_func(
        model, train_mol_ds, test_mol_ds, 1.5
    )
    split_results = test_df.loc[
        :, ["smiles", "parent_remaining_24", "metabolite_detected", "target"]
    ].copy()
    split_results["preds"] = preds
    split_results["pred_probs"] = pred_probs
    split_results["labels"] = labels
    split_results["outer_idx"] = outer_idx
    split_results["inner_idx"] = inner_idx

    wandb_table = wandb.Table(dataframe=split_results, log_mode="INCREMENTAL")
    run.log({"split_results": wandb_table}, step=idx)

wandb.finish()